# Coffee Quality Prediction: A CRISP-DM Approach

This notebook implements a data science project following the **CRISP-DM (Cross-Industry Standard Process for Data Mining)** framework to build a regression-based decision-support tool for coffee quality evaluation.

**Target Variable:** Total Cup Score  
**Data Source:** Coffee Quality Institute (CQI)  
**Models:** Linear Regression, Random Forest Regressor  
**Evaluation Metrics:** MAE, RMSE, R²

---
## Phase 1: Business Understanding

Coffee quality evaluation is traditionally performed through manual cupping by trained professionals. While accurate, this process is time-intensive and does not scale efficiently when evaluating large volumes of coffee samples.

**Business Objective:** Develop a predictive model that estimates Total Cup Score from measurable coffee attributes, enabling faster prioritization of coffee samples for expert evaluation.

**Hypothesis:**
- **H0 (Null):** Coffee attributes do not significantly predict Total Cup Score.
- **H1 (Alternative):** Coffee attributes can significantly predict Total Cup Score.

This model is intended as a decision-support tool, not a replacement for professional cupping.

---
## Phase 2: Data Understanding

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy import stats

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

### 2.1 Load Dataset

In [ ]:
url = 'https://raw.githubusercontent.com/jldbc/coffee-quality-database/master/data/arabica_data_cleaned.csv'
df_raw = pd.read_csv(url)
print(f'Dataset shape: {df_raw.shape}')
df_raw.head()

### 2.2 Initial Exploration

In [ ]:
df_raw.info()

In [ ]:
df_raw.describe()

In [ ]:
print('Missing values per column:')
missing = df_raw.isnull().sum()
print(missing[missing > 0].sort_values(ascending=False))

### 2.3 Define Key Columns

The CQI dataset contains **sensory attributes** scored by professional cuppers as well as **physical/geographic measurements**. The target variable, Total Cup Score (`Total.Cup.Points`), is the sum of individual sensory scores plus a base value.

In [ ]:
sensory_cols = [
    'Aroma', 'Flavor', 'Aftertaste', 'Acidity', 'Body',
    'Balance', 'Uniformity', 'Clean.Cup', 'Sweetness', 'Cupper.Points'
]

physical_cols = ['Moisture', 'Category.One.Defects', 'Category.Two.Defects', 'Quakers']

target_col = 'Total.Cup.Points'

print('Sensory columns:', sensory_cols)
print('Physical columns:', physical_cols)
print('Target:', target_col)

### 2.4 Target Variable Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df_raw[target_col].dropna(), bins=30, color='#8B4513', edgecolor='white', alpha=0.85)
axes[0].set_xlabel('Total Cup Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Total Cup Score')

axes[1].boxplot(df_raw[target_col].dropna(), vert=True, patch_artist=True,
                boxprops=dict(facecolor='#D2B48C'))
axes[1].set_ylabel('Total Cup Score')
axes[1].set_title('Total Cup Score Box Plot')

plt.tight_layout()
plt.show()

print(f'Mean: {df_raw[target_col].mean():.2f}')
print(f'Median: {df_raw[target_col].median():.2f}')
print(f'Std Dev: {df_raw[target_col].std():.2f}')

### 2.5 Sensory Attribute Distributions

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()

for i, col in enumerate(sensory_cols):
    axes[i].hist(df_raw[col].dropna(), bins=20, color='#A0522D', edgecolor='white', alpha=0.8)
    axes[i].set_title(col, fontsize=10)
    axes[i].tick_params(labelsize=8)

plt.suptitle('Distribution of Sensory Attributes', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### 2.6 Correlation Analysis

In [ ]:
corr_cols = sensory_cols + physical_cols + [target_col]
corr_df = df_raw[corr_cols].dropna()
corr_matrix = corr_df.corr()

fig, ax = plt.subplots(figsize=(12, 9))
cax = ax.matshow(corr_matrix, cmap='RdBr', vmin=-1, vmax=1)
fig.colorbar(cax, shrink=0.8)
ax.set_xticks(range(len(corr_cols)))
ax.set_yticks(range(len(corr_cols)))
ax.set_xticklabels(corr_cols, rotation=45, ha='left', fontsize=8)
ax.set_yticklabels(corr_cols, fontsize=8)
ax.set_title('Correlation Matrix: Sensory, Physical, and Target', pad=50)
plt.tight_layout()
plt.show()

In [ ]:
print('Correlations with Total Cup Score:')
print(corr_matrix[target_col].drop(target_col).sort_values(ascending=False).to_string())

---
## Phase 3: Data Preparation

### 3.1 Select and Clean Relevant Columns

In [ ]:
keep_cols = sensory_cols + physical_cols + ['Country.of.Origin', 'altitude_mean_meters'] + [target_col]
df = df_raw[keep_cols].copy()

print(f'Shape before cleaning: {df.shape}')
print(f'Missing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}')

In [ ]:
for col in df.select_dtypes(include=[np.number]).columns:
    df[col] = df[col].fillna(df[col].median())

df['Country.of.Origin'] = df['Country.of.Origin'].fillna('Unknown')

print(f'Shape after cleaning: {df.shape}')
print(f'Remaining missing values: {df.isnull().sum().sum()}')

### 3.2 Handle Outliers

Remove extreme outliers in the target variable that may represent data entry errors or non-representative samples.

In [ ]:
q1 = df[target_col].quantile(0.01)
q99 = df[target_col].quantile(0.99)
before = len(df)
df = df[(df[target_col] >= q1) & (df[target_col] <= q99)]
print(f'Removed {before - len(df)} outlier rows (1st/99th percentile filter)')
print(f'Final shape: {df.shape}')

### 3.3 Encode Categorical Variables

In [ ]:
top_countries = df['Country.of.Origin'].value_counts().nlargest(10).index.tolist()
df['Country_Group'] = df['Country.of.Origin'].apply(
    lambda x: x if x in top_countries else 'Other'
)

country_dummies = pd.get_dummies(df['Country_Group'], prefix='Country', drop_first=True)
df = pd.concat([df, country_dummies], axis=1)
df = df.drop(columns=['Country.of.Origin', 'Country_Group'])

print(f'Shape after encoding: {df.shape}')
print(f'Columns: {list(df.columns)}')

### 3.4 Construct Feature Sets

Two feature approaches are used:

1. **Full Feature Set** — includes all available attributes (sensory + physical + geographic)
2. **Reduced Feature Set** — excludes sensory scoring variables to evaluate the impact of target leakage

Sensory scores (Aroma, Flavor, etc.) are components of Total Cup Score. Including them demonstrates strong predictive performance but involves partial target leakage. The reduced set evaluates whether non-sensory attributes alone carry predictive value.

In [ ]:
feature_cols_full = [c for c in df.columns if c != target_col]
feature_cols_reduced = [c for c in df.columns if c != target_col and c not in sensory_cols]

print(f'Full feature set: {len(feature_cols_full)} features')
print(f'Reduced feature set: {len(feature_cols_reduced)} features')
print(f'\nReduced features: {feature_cols_reduced}')

### 3.5 Train/Test Split

In [ ]:
X_full = df[feature_cols_full]
X_reduced = df[feature_cols_reduced]
y = df[target_col]

X_full_train, X_full_test, y_train, y_test = train_test_split(
    X_full, y, test_size=0.2, random_state=42
)

X_red_train, X_red_test, _, _ = train_test_split(
    X_reduced, y, test_size=0.2, random_state=42
)

print(f'Training samples: {len(X_full_train)}')
print(f'Testing samples: {len(X_full_test)}')

---
## Phase 4: Modeling

Two supervised learning algorithms are applied:

- **Linear Regression:** A baseline model that assumes a linear relationship between features and the target.
- **Random Forest Regressor:** An ensemble method that captures non-linear relationships through decision tree aggregation.

### 4.1 Model Training

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
}

results = []

for model_name, model in models.items():
    for feature_set_name, X_tr, X_te in [
        ('Full', X_full_train, X_full_test),
        ('Reduced', X_red_train, X_red_test)
    ]:
        model_clone = model.__class__(**model.get_params())
        model_clone.fit(X_tr, y_train)
        y_pred = model_clone.predict(X_te)

        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)

        results.append({
            'Model': model_name,
            'Features': feature_set_name,
            'MAE': mae,
            'RMSE': rmse,
            'R2': r2,
            'fitted_model': model_clone,
            'y_pred': y_pred
        })

        print(f'{model_name} ({feature_set_name}): MAE={mae:.4f}, RMSE={rmse:.4f}, R²={r2:.4f}')

---
## Phase 5: Evaluation

### 5.1 Performance Comparison

In [ ]:
results_df = pd.DataFrame(results).drop(columns=['fitted_model', 'y_pred'])
results_df['Label'] = results_df['Model'] + ' (' + results_df['Features'] + ')'
results_df = results_df.set_index('Label')
results_df[['MAE', 'RMSE', 'R2']]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
labels = [r['Model'] + '\n(' + r['Features'] + ')' for r in results]
colors = ['#8B4513', '#D2691E', '#A0522D', '#DEB887']

axes[0].barh(labels, [r['MAE'] for r in results], color=colors)
axes[0].set_xlabel('MAE')
axes[0].set_title('Mean Absolute Error')
axes[0].invert_yaxis()

axes[1].barh(labels, [r['RMSE'] for r in results], color=colors)
axes[1].set_xlabel('RMSE')
axes[1].set_title('Root Mean Squared Error')
axes[1].invert_yaxis()

axes[2].barh(labels, [r['R2'] for r in results], color=colors)
axes[2].set_xlabel('R²')
axes[2].set_title('R² Score')
axes[2].invert_yaxis()

plt.suptitle('Model Performance Comparison', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### 5.2 Actual vs. Predicted

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, r in enumerate(results):
    ax = axes[i]
    ax.scatter(y_test, r['y_pred'], alpha=0.4, s=15, color=colors[i])
    mn, mx = y_test.min(), y_test.max()
    ax.plot([mn, mx], [mn, mx], 'k--', linewidth=1, label='Perfect Prediction')
    ax.set_xlabel('Actual Total Cup Score')
    ax.set_ylabel('Predicted Total Cup Score')
    ax.set_title(f"{r['Model']} ({r['Features']})\nR²={r['R2']:.4f}")
    ax.legend(fontsize=8)

plt.suptitle('Actual vs. Predicted Total Cup Score', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### 5.3 Residual Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, r in enumerate(results):
    residuals = y_test - r['y_pred']
    ax = axes[i]
    ax.scatter(r['y_pred'], residuals, alpha=0.4, s=15, color=colors[i])
    ax.axhline(y=0, color='black', linestyle='--', linewidth=1)
    ax.set_xlabel('Predicted Total Cup Score')
    ax.set_ylabel('Residual')
    ax.set_title(f"{r['Model']} ({r['Features']})")

plt.suptitle('Residual Plots', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### 5.4 Hypothesis Testing

An F-test is used to determine whether the regression model is statistically significant — that is, whether the model explains a significant portion of variance in Total Cup Score compared to a mean-only baseline.

- **H0:** Coffee attributes do not significantly predict Total Cup Score (model is no better than the mean).
- **H1:** Coffee attributes can significantly predict Total Cup Score.

In [ ]:
best_full = [r for r in results if r['Features'] == 'Full'][0]
best_reduced = [r for r in results if r['Features'] == 'Reduced'][0]

for label, r, X_test in [
    ('Full Feature Set', best_full, X_full_test),
    ('Reduced Feature Set', best_reduced, X_red_test)
]:
    y_pred = r['y_pred']
    n = len(y_test)
    p = X_test.shape[1]

    ss_res = np.sum((y_test - y_pred) ** 2)
    ss_tot = np.sum((y_test - y_test.mean()) ** 2)
    ss_reg = ss_tot - ss_res

    df_reg = p
    df_res = n - p - 1

    if df_res > 0 and ss_res > 0:
        ms_reg = ss_reg / df_reg
        ms_res = ss_res / df_res
        f_stat = ms_reg / ms_res
        p_value = 1 - stats.f.cdf(f_stat, df_reg, df_res)
    else:
        f_stat = np.inf
        p_value = 0.0

    print(f'--- {r["Model"]} ({label}) ---')
    print(f'F-statistic: {f_stat:.4f}')
    print(f'p-value: {p_value:.2e}')
    if p_value < 0.05:
        print('Result: Reject H0. Coffee attributes significantly predict Total Cup Score.')
    else:
        print('Result: Fail to reject H0.')
    print()

### 5.5 Feature Importance (Random Forest — Full Feature Set)

In [ ]:
rf_full = [r for r in results if r['Model'] == 'Random Forest' and r['Features'] == 'Full'][0]
importances = rf_full['fitted_model'].feature_importances_
feat_imp = pd.Series(importances, index=feature_cols_full).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, max(6, len(feat_imp) * 0.3)))
feat_imp.plot(kind='barh', color='#8B4513', ax=ax)
ax.set_xlabel('Feature Importance')
ax.set_title('Random Forest Feature Importance (Full Feature Set)')
plt.tight_layout()
plt.show()

### 5.6 Feature Importance (Random Forest — Reduced Feature Set)

In [ ]:
rf_red = [r for r in results if r['Model'] == 'Random Forest' and r['Features'] == 'Reduced'][0]
importances_red = rf_red['fitted_model'].feature_importances_
feat_imp_red = pd.Series(importances_red, index=feature_cols_reduced).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, max(6, len(feat_imp_red) * 0.3)))
feat_imp_red.plot(kind='barh', color='#A0522D', ax=ax)
ax.set_xlabel('Feature Importance')
ax.set_title('Random Forest Feature Importance (Reduced Feature Set)')
plt.tight_layout()
plt.show()

### 5.7 Key Findings

**Full Feature Set:**
- Models achieve high R² values, confirming that sensory attributes are strong predictors of Total Cup Score.
- This is expected because sensory scores are components of the target variable (partial target leakage).
- The full-feature model demonstrates the relationship between individual scores and overall quality.

**Reduced Feature Set:**
- Without sensory attributes, predictive accuracy decreases substantially.
- This confirms that physical and geographic attributes alone carry limited predictive power for Total Cup Score.
- The reduced model is more realistic for scenarios where cupping has not yet been performed.

**Hypothesis Conclusion:**
- The F-test results reject the null hypothesis (H0), confirming that coffee attributes significantly predict Total Cup Score in both feature configurations.

---
## Phase 6: Deployment (Decision Support System)

The final data product is a prediction function that accepts coffee attributes and returns an estimated Total Cup Score. This is intended for exploratory decision support — prioritizing coffee samples for expert evaluation rather than replacing the cupping process.

In [ ]:
final_model = rf_full['fitted_model']
final_features = feature_cols_full


def predict_cup_score(sample_dict):
    """Estimate Total Cup Score from coffee attributes."""
    sample_df = pd.DataFrame([sample_dict])
    for col in final_features:
        if col not in sample_df.columns:
            sample_df[col] = 0
    sample_df = sample_df[final_features]
    prediction = final_model.predict(sample_df)[0]
    return round(prediction, 2)

In [ ]:
sample_coffee = {
    'Aroma': 7.75, 'Flavor': 7.50, 'Aftertaste': 7.25,
    'Acidity': 7.50, 'Body': 7.50, 'Balance': 7.50,
    'Uniformity': 10.0, 'Clean.Cup': 10.0, 'Sweetness': 10.0,
    'Cupper.Points': 7.50, 'Moisture': 0.10,
    'Category.One.Defects': 0, 'Category.Two.Defects': 2,
    'Quakers': 0, 'altitude_mean_meters': 1500
}

predicted_score = predict_cup_score(sample_coffee)
print(f'Sample coffee attributes: {sample_coffee}')
print(f'Predicted Total Cup Score: {predicted_score}')

---
## Conclusion

This project followed the CRISP-DM framework to develop a regression-based decision-support tool for coffee quality evaluation.

**Key outcomes:**

1. **Business Understanding:** The need for faster coffee quality screening was identified and framed as a predictive modeling problem.
2. **Data Understanding:** The CQI dataset was explored, revealing strong correlations between sensory attributes and Total Cup Score.
3. **Data Preparation:** Missing values were handled, categorical variables were encoded, and two feature sets were constructed to evaluate the impact of target leakage.
4. **Modeling:** Linear Regression and Random Forest models were trained on both feature sets.
5. **Evaluation:** Models with sensory features achieved high accuracy (expected due to partial target leakage). The reduced feature set demonstrated more realistic but limited predictive performance. The null hypothesis was rejected in both configurations.
6. **Deployment:** A prediction function was implemented for exploratory decision support.

**Limitations:**
- Partial target leakage exists in the full feature set, as sensory scores are components of Total Cup Score.
- The reduced feature set has limited predictive power, reflecting the inherent difficulty of predicting quality without sensory evaluation.
- This system is a decision-support tool and does not replace professional cupping.